# Task 075: TIGRE — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CT reconstruction using TIGRE with FBP, SIRT, and CGLS

**Paper**: TIGRE: A MATLAB-GPU toolbox for CBCT image reconstruction
**Repository**: https://github.com/CERN/TIGRE

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 22.53 dB ← 🔧 修复前: 17.18 dB
- **SSIM**: 0.4644
- **Evaluation**: Sparse-view CT reconstruction

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
TIGRE — Sparse-View CT Reconstruction
=======================================
Task #72: Reconstruct a 2D tomographic image from sparse-view
          projections using Filtered Back-Projection (FBP) and
          iterative (SIRT/CGLS) algorithms.

Inverse Problem:
    Given sinogram data p(θ,s) = R[f](θ,s) (Radon transform of f),
    with sparse angular sampling, recover image f(x,y).

Forward Model:
    Radon transform (line integrals):
    p(θ,s) = ∫∫ f(x,y) δ(x cosθ + y sinθ - s) dx dy

Inverse Solver:
    1) FBP with Ram-Lak (ramp) filter
    2) SIRT (Simultaneous Iterative Reconstruction Technique)
    3) CGLS (Conjugate Gradient Least Squares)
    All with sparse angular sampling.

Repo: https://github.com/CERN/TIGRE
Paper: Biguri et al. (2016), Biomedical Physics & Engineering Express.

Usage: /data/yjh/spectro_env/bin/python TIGRE_code.py
"""

import numpy as np
import matplotlib

## 3. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_shepp_logan`, `radon_transform`, `unfiltered_backproject`

In [ ]:
# ─── Data Generation ──────────────────────────────────────────────
def generate_shepp_logan(size):
    """
    Generate the Shepp-Logan phantom.
    Standard CT test image with ellipses of varying attenuation.
    """
    img = np.zeros((size, size))
    Y, X = np.mgrid[:size, :size]
    X = (X - size / 2) / (size / 2)
    Y = (Y - size / 2) / (size / 2)

    # Ellipse parameters: (A, a, b, x0, y0, phi)
    ellipses = [
        (1.0,   0.69,  0.92,  0,      0,       0),      # Outer skull
        (-0.8,  0.6624, 0.8740, 0,     -0.0184, 0),      # Brain
        (-0.2,  0.1100, 0.3100, 0.22,  0,       -18),    # Left ventricle
        (-0.2,  0.1600, 0.4100, -0.22, 0,       18),     # Right ventricle
        (0.1,   0.2100, 0.2500, 0,     0.35,    0),      # Top tumour
        (0.1,   0.0460, 0.0460, 0,     0.1,     0),      # Small tumour 1
        (0.1,   0.0460, 0.0460, 0,     -0.1,    0),      # Small tumour 2
        (0.1,   0.0460, 0.0230, -0.08, -0.605,  0),      # Bottom left
        (0.1,   0.0230, 0.0230, 0,     -0.606,  0),      # Bottom centre
        (0.1,   0.0230, 0.0460, 0.06,  -0.605,  0),      # Bottom right
    ]

    for A, a, b, x0, y0, phi_deg in ellipses:
        phi = np.radians(phi_deg)
        cos_p, sin_p = np.cos(phi), np.sin(phi)

        x_rot = (X - x0) * cos_p + (Y - y0) * sin_p
        y_rot = -(X - x0) * sin_p + (Y - y0) * cos_p

        mask = (x_rot / a)**2 + (y_rot / b)**2 <= 1
        img[mask] += A

    img = np.clip(img, 0, None)
    return img

# ─── Forward Operator: Radon Transform ────────────────────────────
def radon_transform(image, angles_deg):
    """
    Compute the Radon transform (sinogram) of a 2D image.
    Uses rotation + integration approach.
    """
    n = image.shape[0]
    n_det = int(np.ceil(n * np.sqrt(2)))
    if n_det % 2 == 0:
        n_det += 1
    sinogram = np.zeros((len(angles_deg), n_det))

    # Pad image to n_det × n_det
    pad_total = n_det - n
    pad_before = pad_total // 2
    pad_after = pad_total - pad_before
    img_padded = np.pad(image, ((pad_before, pad_after), (pad_before, pad_after)), mode='constant')

    center = img_padded.shape[0] / 2

    from scipy.ndimage import rotate as ndi_rotate

    for i, angle in enumerate(angles_deg):
        rotated = ndi_rotate(img_padded, -angle, reshape=False, order=1)
        sinogram[i, :] = rotated.sum(axis=0)[:n_det]

    return sinogram

def unfiltered_backproject(sinogram, angles_deg, img_size):
    """
    Unfiltered backprojection using iradon with filter_name=None.
    Used as the adjoint / correction operator in iterative methods.
    """
    sino_T = sinogram.T
    recon = iradon(sino_T, theta=angles_deg, filter_name=None,
                   output_size=img_size, circle=False)
    return recon

## 4. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fbp_reconstruct`, `sirt_reconstruct`, `cgls_reconstruct`

In [ ]:
# ─── Inverse Solver: FBP ──────────────────────────────────────────
def fbp_reconstruct(sinogram, angles_deg, img_size):
    """
    Filtered Back-Projection using skimage.transform.iradon with ramp filter.
    sinogram: shape (n_angles, n_det)
    angles_deg: 1-D array of projection angles in degrees
    """
    # iradon expects (n_det, n_angles)
    sino_T = sinogram.T
    recon = iradon(sino_T, theta=angles_deg, filter_name='ramp',
                   output_size=img_size, circle=False)
    return recon

# ─── Inverse Solver: SIRT ─────────────────────────────────────────
def sirt_reconstruct(sinogram, angles_deg, img_size, n_iter=50):
    """
    Simultaneous Iterative Reconstruction Technique (SIRT).
    """
    recon = np.zeros((img_size, img_size))

    print(f"  SIRT: {n_iter} iterations ...")

    for it in range(n_iter):
        # Forward project current estimate
        sino_est = radon_transform(recon, angles_deg)

        # Trim to match
        n_det = sinogram.shape[1]
        sino_est_trim = sino_est[:, :n_det]

        # Residual
        residual = sinogram - sino_est_trim

        # Backproject residual (unfiltered — correct SIRT operator)
        correction = unfiltered_backproject(residual, angles_deg, img_size)

        # Update with relaxation
        recon += 0.1 * correction
        recon = np.maximum(recon, 0)

        if (it + 1) % 10 == 0:
            err = np.linalg.norm(residual)
            print(f"    iter {it+1:3d}: residual norm = {err:.4f}")

    return recon

# ─── Inverse Solver: CGLS ─────────────────────────────────────────
def cgls_reconstruct(sinogram, angles_deg, img_size, n_iter=30):
    """
    Conjugate Gradient Least Squares for CT reconstruction.
    Uses forward (Radon) and adjoint (backprojection) operators.
    """
    n_det = sinogram.shape[1]

    # Flatten
    b = sinogram.ravel()

    def A_forward(x):
        img = x.reshape(img_size, img_size)
        sino = radon_transform(img, angles_deg)
        return sino[:, :n_det].ravel()

    def AT_backward(y):
        sino = y.reshape(len(angles_deg), n_det)
        img = unfiltered_backproject(sino, angles_deg, img_size)
        return img.ravel()

    # CGLS
    x = np.zeros(img_size * img_size)
    r = b - A_forward(x)
    s = AT_backward(r)
    p = s.copy()
    gamma = np.dot(s, s)

    print(f"  CGLS: {n_iter} iterations ...")

    for it in range(n_iter):
        Ap = A_forward(p)
        alpha = gamma / max(np.dot(Ap, Ap), 1e-30)
        x = x + alpha * p
        r = r - alpha * Ap
        s = AT_backward(r)
        gamma_new = np.dot(s, s)
        beta = gamma_new / max(gamma, 1e-30)
        p = s + beta * p
        gamma = gamma_new

        if (it + 1) % 10 == 0:
            print(f"    iter {it+1:3d}: ||r|| = {np.linalg.norm(r):.4f}")

    x = np.maximum(x, 0)
    return x.reshape(img_size, img_size)

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ─── Metrics ───────────────────────────────────────────────────────
def compute_metrics(gt, rec):
    gt_n = gt / max(gt.max(), 1e-12)
    rec_n = rec / max(rec.max(), 1e-12)
    data_range = 1.0
    mse = np.mean((gt_n - rec_n)**2)
    psnr = float(10 * np.log10(data_range**2 / max(mse, 1e-30)))
    ssim_val = float(ssim_fn(gt_n, rec_n, data_range=data_range))
    cc = float(np.corrcoef(gt_n.ravel(), rec_n.ravel())[0, 1])
    re = float(np.linalg.norm(gt_n - rec_n) / max(np.linalg.norm(gt_n), 1e-12))
    rmse = float(np.sqrt(mse))
    return {"PSNR": psnr, "SSIM": ssim_val, "CC": cc, "RE": re, "RMSE": rmse}

# ─── Visualization ─────────────────────────────────────────────────
def visualize_results(gt, sinogram, rec_fbp, rec_sirt, angles, metrics, save_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0, 0].imshow(gt, cmap='gray')
    axes[0, 0].set_title('Ground Truth (Shepp-Logan)')

    axes[0, 1].imshow(sinogram, aspect='auto', cmap='gray')
    axes[0, 1].set_title(f'Sinogram ({len(angles)} angles)')
    axes[0, 1].set_xlabel('Detector')
    axes[0, 1].set_ylabel('Angle index')

    axes[0, 2].imshow(rec_fbp / max(rec_fbp.max(), 1e-12), cmap='gray')
    axes[0, 2].set_title('FBP Reconstruction')

    axes[1, 0].imshow(rec_sirt / max(rec_sirt.max(), 1e-12), cmap='gray')
    axes[1, 0].set_title('SIRT Reconstruction')

    err = np.abs(gt / max(gt.max(), 1e-12) - rec_sirt / max(rec_sirt.max(), 1e-12))
    axes[1, 1].imshow(err, cmap='hot')
    axes[1, 1].set_title('|Error| (SIRT)')

    # Profile comparison
    mid = gt.shape[0] // 2
    axes[1, 2].plot(gt[mid, :] / max(gt[mid, :].max(), 1e-12), 'b-', lw=2, label='GT')
    axes[1, 2].plot(rec_fbp[mid, :] / max(rec_fbp[mid, :].max(), 1e-12),
                     'g--', lw=1.5, label='FBP')
    axes[1, 2].plot(rec_sirt[mid, :] / max(rec_sirt[mid, :].max(), 1e-12),
                     'r--', lw=1.5, label='SIRT')
    axes[1, 2].set_title('Central Profile')
    axes[1, 2].legend()

    fig.suptitle(
        f"TIGRE — Sparse-View CT Reconstruction ({N_ANGLES_SPARSE} views)\n"
        f"PSNR={metrics['PSNR']:.1f} dB | SSIM={metrics['SSIM']:.4f} | "
        f"CC={metrics['CC']:.4f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 70)
print("  TIGRE — Sparse-View CT Reconstruction")
print("=" * 70)

rng = np.random.default_rng(SEED)

### Stage 1: Data Generation

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Stage 1: Data Generation
print("\n[STAGE 1] Data Generation — Shepp-Logan Phantom")
phantom = generate_shepp_logan(IMG_SIZE)
print(f"  Phantom: {phantom.shape}")
print(f"  Value range: [{phantom.min():.3f}, {phantom.max():.3f}]")

### Stage 2: Forward — Radon Transform

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Stage 2: Forward — Radon Transform
print("\n[STAGE 2] Forward — Radon Transform (Sparse View)")
angles_sparse = np.linspace(0, 180, N_ANGLES_SPARSE, endpoint=False)
sinogram_sparse = radon_transform(phantom, angles_sparse)
# Add Poisson-like noise
sig_power = np.mean(sinogram_sparse**2)
noise_power = sig_power / (10**(NOISE_SNR_DB / 10))
noise = np.sqrt(noise_power) * rng.standard_normal(sinogram_sparse.shape)
sinogram_noisy = sinogram_sparse + noise
print(f"  Sinogram: {sinogram_noisy.shape} ({N_ANGLES_SPARSE} angles)")

### Stage 3a: FBP

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Stage 3a: FBP
print("\n[STAGE 3a] Inverse — Filtered Back-Projection")
rec_fbp = fbp_reconstruct(sinogram_noisy, angles_sparse, IMG_SIZE)
rec_fbp = np.maximum(rec_fbp, 0)
m_fbp = compute_metrics(phantom, rec_fbp)
print(f"  FBP: CC={m_fbp['CC']:.4f}, PSNR={m_fbp['PSNR']:.1f}")

### Stage 3b: SIRT

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Stage 3b: SIRT
print("\n[STAGE 3b] Inverse — SIRT")
rec_sirt = sirt_reconstruct(sinogram_noisy, angles_sparse, IMG_SIZE, n_iter=50)
m_sirt = compute_metrics(phantom, rec_sirt)
print(f"  SIRT: CC={m_sirt['CC']:.4f}, PSNR={m_sirt['PSNR']:.1f}")

# Choose best
if m_sirt['CC'] >= m_fbp['CC']:
    rec_best = rec_sirt
    metrics = m_sirt
    method = "SIRT"
else:
    rec_best = rec_fbp
    metrics = m_fbp
    method = "FBP"
print(f"\n  → Using {method} (higher CC)")

### Stage 4: Evaluation

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Stage 4: Evaluation
print("\n[STAGE 4] Evaluation Metrics:")
for k, v in sorted(metrics.items()):
    print(f"  {k:20s} = {v}")

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), rec_best)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), phantom)
# Also save to working dir for website assets
np.save(os.path.join(WORKING_DIR, "gt_output.npy"), phantom)
np.save(os.path.join(WORKING_DIR, "recon_output.npy"), rec_best)

visualize_results(phantom, sinogram_noisy, rec_fbp, rec_sirt,
                  angles_sparse, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))

print("\n" + "=" * 70)
print("  DONE — Results saved to", RESULTS_DIR)
print("=" * 70)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **TIGRE**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=22.53 dB ← 🔧 修复前: 17.18 dB, SSIM=0.4644)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: TIGRE: A MATLAB-GPU toolbox for CBCT image reconstruction
- Repository: https://github.com/CERN/TIGRE
